In [ ]:
from pathlib import Path
import yaml
import tqdm
import pandas as pd
import numpy as np
from rdkit import Chem

from benchmark_data_leakage.chembl_requester import ChEMBLRequester
from benchmark_data_leakage.utils import find_repo_root, get_canonical_smiles_from_smiles

repo_root = find_repo_root()
data_dir  = repo_root / "data"
out_dir   = data_dir / "out"
out_dir.mkdir(exist_ok=True, parents=True)

# NOTE: Set this to the root of the IndustryBenchmarks2024 repo
# https://github.com/OpenFreeEnergy/IndustryBenchmarks2024
# commit ff41e5ad0fda89b352341e3f6511bee25db0000a
industry_benchmarks_root =

TARGET_MAPPING_OPEN_FE = {
    # Keys are `system group` and `system name` as in this file https://github.com/OpenFreeEnergy/IndustryBenchmarks2024/blob/ff41e5ad0fda89b352341e3f6511bee25db0000a/industry_benchmarks/analysis/processed_results/combined_pymbar3_calculated_dg_data.csv
    # Values are uniprot_id
    # Mapped performed by looking at https://github.com/OpenFreeEnergy/IndustryBenchmarks2024/tree/ff41e5ad0fda89b352341e3f6511bee25db0000a/industry_benchmarks/input_structures/original_structures/*/subset_metadata.csv
    # and mapping `Reference PDB` column to uniprot_id
    ('charge_annihilation_set', 'cdk2'): 'P24941',
    ('charge_annihilation_set', 'dlk'): 'Q12852',
    ('charge_annihilation_set', 'egfr'): 'P00533',
    ('charge_annihilation_set', 'ephx2'): 'P34913',
    ('charge_annihilation_set', 'irak4_s2'): 'Q9NWZ3',
    ('charge_annihilation_set', 'irak4_s3'): 'Q9NWZ3',
    ('charge_annihilation_set', 'itk'): 'Q08881',
    ('charge_annihilation_set', 'jak1'): 'P23458',
    ('charge_annihilation_set', 'jnk1'): 'P45983',
    ('charge_annihilation_set', 'ptp1b'): 'P18031',
    ('charge_annihilation_set', 'thrombin'): 'P00734',
    ('charge_annihilation_set', 'tyk2'): 'P29597',
    ('fragments', 'hsp90_2rings'): 'P07900',
    ('fragments', 'hsp90_single_ring'): 'P07900',
    ('fragments', 'jak2_set1'): 'O60674',
    ('fragments', 'jak2_set2'): 'O60674',
    ('fragments', 'liga'): 'Q9AIU7',
    ('fragments', 'mcl1'): 'Q07820',
    ('fragments', 'mup1'): 'P02762',
    ('fragments', 'p38'): 'Q16539',
    ('fragments', 't4_lysozyme'): 'P00720',
    ('jacs_set', 'bace'): 'P56817',
    ('jacs_set', 'cdk2'): 'P24941',
    ('jacs_set', 'jnk1'): 'P45983',
    ('jacs_set', 'mcl1'): 'Q07820',
    ('jacs_set', 'p38'): 'Q16539',
    ('jacs_set', 'ptp1b'): 'P18031',
    ('jacs_set', 'thrombin'): 'P00734',
    ('jacs_set', 'tyk2'): 'P29597',
    ('janssen_bace', 'bace_ciordia_prospective'): 'P56817',
    ('janssen_bace', 'bace_p3_arg368_in'): 'P56817',
    ('janssen_bace', 'ciordia_retro'): 'P56817',
    ('janssen_bace', 'keranen_p2'): 'P56817',
    ('mcs_docking_set', 'hne'): 'P08246',
    ('mcs_docking_set', 'renin'): 'P00797',
    ('merck', 'cdk8'): 'P49336',
    ('merck', 'cmet'): 'P08581',
    ('merck', 'eg5'): 'P52732',
    ('merck', 'hif2a'): 'Q99814',
    ('merck', 'pfkfb3'): 'Q16875',
    ('merck', 'shp2'): 'Q06124',
    ('merck', 'syk'): 'P43405',
    ('merck', 'tnks2'): 'Q9H2K2',
    ('miscellaneous_set', 'btk'): 'Q06187',
    ('miscellaneous_set', 'cdk8'): 'P49336',
    ('miscellaneous_set', 'faah'): 'P97612',
    ('miscellaneous_set', 'galectin'): 'P17931',
    ('miscellaneous_set', 'hiv1_protease'): 'Q5RZ09',
    ('scaffold_hopping_set', 'bace1'): 'P56817',
    ('scaffold_hopping_set', 'factor_xa'): 'P00742',
    ('water_set', 'brd4'): 'O60885',
    ('water_set', 'chk1'): 'O14757',
    ('water_set', 'hsp90_kung'): 'P07900',
    ('water_set', 'hsp90_woodhead'): 'P07900',
    ('water_set', 'scyt_dehyd'): 'P56221',
    ('water_set', 'taf12'): 'P21675',  # Note this is TAF1(2) and not TAF12 as per the subset_metadata.csv
    ('water_set', 'thrombin'): 'P00734',
    ('water_set', 'urokinase'): 'P00749',
}

### Extract OpenFE benchmark data

In [ ]:
csv_path = (
    industry_benchmarks_root
    / "industry_benchmarks/analysis/processed_results"
    / "combined_pymbar3_calculated_dg_data.csv"
)
raw_df = pd.read_csv(csv_path)
raw_df = raw_df.rename(columns={
    "ligand name":       "ligand_name",
    "Exp DG (kcal/mol)": "exp_dg_kcal_mol",
    "Exp dDG (kcal/mol)": "uncertainty_kcal_mol",
    "system group":      "system_group",
    "system name":       "system_name",
})
print(f"Loaded {len(raw_df)} rows, {raw_df['system_name'].nunique()} unique targets")
print(raw_df[["system_group", "system_name"]].drop_duplicates().to_string(index=False))

In [ ]:
RT_LN10 = 0.592 * np.log(10)  # kcal/mol at 298 K; used for DG -> pchembl conversion


def _name_variants(name: str) -> list[str]:
    """Return alternative spellings of an SDF molecule name to match CSV ligand_name.

    The CSV ligand names were derived from SDF mol names by differing conventions:
      - spaces replaced with hyphens  (most targets)
      - spaces replaced with underscores  (merck/cmet)
      - ' out' abbreviated to 'o'  (janssen_bace/keranen_p2)
    Storing all variants lets the lookup succeed regardless of convention.
    """
    variants = {name}
    variants.add(name.replace(" ", "-"))
    variants.add(name.replace(" ", "_"))
    variants.add(name.replace(" out", "o"))
    return list(variants)


def load_smiles_from_sdf(sdf_path: Path) -> dict[str, str]:
    """Return {molecule_name: canonical_smiles} for all valid molecules in an SDF file.

    Each molecule is stored under all name variants to handle CSV naming conventions.
    """
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=True)
    result = {}
    for mol in supplier:
        if mol is None:
            continue
        name = mol.GetProp("_Name").strip()
        smiles = Chem.MolToSmiles(mol)
        for variant in _name_variants(name):
            result[variant] = smiles
    return result


prepared_root = (
    industry_benchmarks_root
    / "industry_benchmarks/input_structures/prepared_structures"
)

# Build (system_group, system_name) -> {ligand_name: canonical_smiles}
sdf_smiles: dict[tuple, dict] = {}
for sg, sn in TARGET_MAPPING_OPEN_FE:
    sdf_path = prepared_root / sg / sn / "ligands.sdf"
    if sdf_path.exists():
        sdf_smiles[(sg, sn)] = load_smiles_from_sdf(sdf_path)
    else:
        print(f"Warning: SDF not found for ({sg}, {sn}) — {sdf_path}")

print(f"Loaded SDF files for {len(sdf_smiles)} / {len(TARGET_MAPPING_OPEN_FE)} systems")

In [ ]:
with open(repo_root / "chembl_db.yaml") as f:
    chembl_db = yaml.safe_load(f)
chembl_req = ChEMBLRequester(**chembl_db)

# Look up gene_name and chembl_target_id from ChEMBL for each UniProt ID
protein_targets = pd.DataFrame(chembl_req.get_single_protein_targets())
uniprot_to_info = (
    protein_targets
    .drop_duplicates("uniprot_id")
    .set_index("uniprot_id")[["gene_name", "chembl_target_id"]]
    .to_dict("index")
)

records = []
n_smiles_missing = 0
for _, row in raw_df.iterrows():
    sg, sn = row["system_group"], row["system_name"]
    uniprot_id = TARGET_MAPPING_OPEN_FE.get((sg, sn))
    if uniprot_id is None:
        continue  # target not in mapping; skip

    smiles_map = sdf_smiles.get((sg, sn), {})
    raw_smiles = smiles_map.get(row["ligand_name"])
    if raw_smiles is not None:
        canonical = get_canonical_smiles_from_smiles(raw_smiles)
    else:
        canonical = None
        n_smiles_missing += 1

    info = uniprot_to_info.get(uniprot_id, {})
    exp_dg = row["exp_dg_kcal_mol"]

    records.append({
        "system_group":        sg,
        "system_name":         sn,
        "assay_id":            f"{sg}-{sn}",
        "ligand_name":         f"{sg}_{sn}_{row['ligand_name']}",  # To make ligand_name unique
        "smiles":              canonical,
        "partner_id":          row["partner_id"],
        "exp_dg_kcal_mol":     exp_dg,
        "uncertainty_kcal_mol": row["uncertainty_kcal_mol"],
        "pchembl_value":       round(-exp_dg / RT_LN10, 2) if pd.notna(exp_dg) else None,
        "gene_name":           info.get("gene_name"),
        "uniprot_id":          uniprot_id,
        "chembl_target_id":    info.get("chembl_target_id"),
    })

benchmark_df = pd.DataFrame(records)
print(f"Assembled {len(benchmark_df)} rows across {benchmark_df['system_name'].nunique()} systems")
print(f"Missing SMILES: {n_smiles_missing}")
benchmark_df.head()

### Add chembl_ligand_id for those that exist in ChEMBL. This takes time!

In [ ]:
chembl_id_to_smiles = chembl_req.get_chembl_id_to_smiles()

smiles_to_chembl_id = {}
for chembl_lig_id, smiles in tqdm.tqdm(chembl_id_to_smiles, desc="Canonicalising smiles"):
    try:
        canonical_smiles = get_canonical_smiles_from_smiles(smiles)
    except ValueError:
        continue
    smiles_to_chembl_id[canonical_smiles] = chembl_lig_id

benchmark_df["chembl_ligand_id"] = benchmark_df["smiles"].map(smiles_to_chembl_id)
print(f"Fraction with chembl_ligand_id: {(~benchmark_df['chembl_ligand_id'].isna()).mean():.2%}")

### Save the data

In [ ]:
out_fp = out_dir / "OpenFE_benchmark.csv"
print(f"Dumping data to {out_fp}")
benchmark_df.to_csv(out_fp, index=False)